# Z3 从零到 CO3：最小但完整的符号求解教程

这份 notebook 的目标不是把 Z3 API 全讲完，而是帮你真正理解 CO3 demo 里主机端在做什么：

```text
具体输入 -> 设备跑出 trace -> 主机创建符号变量 -> 添加路径约束 -> Z3 求下一轮输入
```

学完这里，你应该能看懂同目录下 `host.py` 里这些东西：

- `BitVec("input_0", 8)` 是什么
- `Solver()` 是什么
- `s.add(...)` 是在加什么
- `s.check() == sat` 表示什么
- `model[var].as_long()` 为什么能变成下一轮输入
- `UGE/ULE` 为什么比 `>`、`<=` 更适合协议字节
- “翻转分支”到底是在哪里发生的

## 1. 一句话理解 Z3

Z3 是一个 SMT Solver。你可以先把它理解成：

```text
给它一堆约束，它帮你找一组变量取值，让所有约束同时成立。
```

比如：

```text
x + 5 == 15
```

Z3 会回答：

```text
x = 10
```

在 CO3 里，变量通常不是普通数学变量，而是协议输入字节，例如 `input_0`、`input_1`。

In [1]:
from z3 import *

print("z3 imported")

z3 imported


## 2. 第一个 Solver：变量、约束、模型

Z3 的最小工作流只有四步：

```text
1. 定义变量
2. 创建 Solver
3. 添加约束
4. check + 读取 model
```

`model` 就是 Z3 找到的一组答案。

In [2]:
x = Int("x")

s = Solver()
s.add(x + 5 == 15)

print("check:", s.check())
print("model:", s.model())
print("x =", s.model()[x])

check: sat
model: [x = 10]
x = 10


### 2.1 `sat` / `unsat` / `unknown`

`check()` 常见有三种结果：

| 结果 | 意思 |
|---|---|
| `sat` | satisfiable，有解 |
| `unsat` | unsatisfiable，无解 |
| `unknown` | Z3 无法确定，普通线性/位向量约束里较少遇到 |

看一个无解例子：

In [3]:
x = Int("x")
s = Solver()
s.add(x > 10)
s.add(x < 5)

print("check:", s.check())

check: unsat


### 2.2 常用 Z3 API：`check()`、`model()`、`as_long()`

先把最常见的几行代码拆开看：

```python
result = s.check()
model = s.model()
value = model[x].as_long()
```

它们分别表示：

| API | 含义 |
|---|---|
| `s.check()` | 让 Z3 开始求解当前 solver 里的所有约束 |
| `sat` | 有解，可以读取 `model` |
| `unsat` | 无解，不应该读取普通 model |
| `s.model()` | 取得 Z3 找到的一组变量取值 |
| `model[x]` | 读取变量 `x` 在这组解里的 Z3 值 |
| `.as_long()` | 把 Z3 的整数/位向量值转成 Python 的 `int` |

最重要的顺序是：**先 `check()`，确认是 `sat`，再读 `model()`。**

In [4]:
x = Int("x")

s = Solver()
s.add(x == 42)

result = s.check()
print("1. check() 返回:", result)

if result == sat:
    model = s.model()
    print("2. model() 返回:", model)

    z3_value = model[x]
    print("3. model[x] 返回:", z3_value, "类型:", type(z3_value))

    py_value = z3_value.as_long()
    print("4. as_long() 返回:", py_value, "类型:", type(py_value))

1. check() 返回: sat
2. model() 返回: [x = 42]
3. model[x] 返回: 42 类型: <class 'z3.z3.IntNumRef'>
4. as_long() 返回: 42 类型: <class 'int'>


#### `check()` 到底检查什么？

`check()` 检查的是当前 `Solver` 里累计的全部约束。

```python
s.add(x > 0)
s.add(x < 10)
s.check()
```

意思是问 Z3：

```text
是否存在一个 x，让 x > 0 且 x < 10 同时成立？
```

所以 `check()` 不是检查某一个变量，也不是执行程序；它是在检查“约束集合是否有解”。

In [5]:
x = Int("x")
s = Solver()

s.add(x > 0)
print("only x > 0:", s.check(), s.model())

s.add(x < 10)
print("x > 0 and x < 10:", s.check(), s.model())

s.add(x > 100)
print("x > 0 and x < 10 and x > 100:", s.check())

only x > 0: sat [x = 1]
x > 0 and x < 10: sat [x = 1]
x > 0 and x < 10 and x > 100: unsat


#### 为什么要用 `as_long()`？

`model[x]` 拿到的不是 Python 原生数字，而是 Z3 自己的数值对象。

这在打印时看起来像 `42`，但如果你要把它放回协议输入、写入 bytearray、传给设备，最好转成 Python `int`：

```python
next_input[idx] = model[var].as_long()
```

这就是 `host.py` 里用 `as_long()` 的原因。

In [6]:
b = BitVec("b", 8)

s = Solver()
s.add(b == 0x5A)
s.check()
m = s.model()

z3_value = m[b]
py_value = z3_value.as_long()

print("Z3 value:", z3_value, type(z3_value))
print("Python int:", py_value, type(py_value))
print("hex:", hex(py_value))

# Python int 可以直接放进 bytearray / dict / 设备输入
packet = bytearray([0, 0])
packet[0] = py_value
print("packet:", packet.hex())

Z3 value: 90 <class 'z3.z3.BitVecNumRef'>
Python int: 90 <class 'int'>
hex: 0x5a
packet: 5a00


#### `model[var]` 可能是 `None`

如果变量没有被任何约束影响，Z3 不一定会在 model 里给它赋值。

这就是为什么 CO3 demo 里会这样写：

```python
value = model[var]
if value is not None:
    next_input[idx] = value.as_long()
```

意思是：Z3 求出来哪个字段，就更新哪个字段；没求出来的字段，沿用当前输入。

In [7]:
input_0 = BitVec("input_0", 8)
input_1 = BitVec("input_1", 8)

s = Solver()
s.add(input_0 == 0x5A)

print("check:", s.check())
m = s.model()
print("model:", m)
print("model[input_0]:", m[input_0])
print("model[input_1]:", m[input_1])

current_input = {0: 0x10, 1: 0x09}
next_input = dict(current_input)

for idx, var in [(0, input_0), (1, input_1)]:
    value = m[var]
    if value is not None:
        next_input[idx] = value.as_long()

print("current_input:", current_input)
print("next_input:", next_input)

check: sat
model: [input_0 = 90]
model[input_0]: 90
model[input_1]: None
current_input: {0: 16, 1: 9}
next_input: {0: 90, 1: 9}


#### `as_long()` 和 BitVec 的关系

对 `BitVec("b", 8)` 来说，`as_long()` 返回的是这个 8-bit 值的无符号整数解释，范围是 `0 ~ 255`。

例如 `0xFF` 会转成 `255`，不是 `-1`。

In [8]:
b = BitVec("b", 8)
s = Solver()
s.add(b == 0xFF)
s.check()

v = s.model()[b]
print("Z3 BitVec value:", v)
print("as_long():", v.as_long())
print("hex:", hex(v.as_long()))

Z3 BitVec value: 255
as_long(): 255
hex: 0xff


## 3. 多个变量：约束是一起成立的

Z3 不是逐条单独求解，而是让所有 `add()` 进去的约束同时成立。

In [9]:
x = Int("x")
y = Int("y")

s = Solver()
s.add(x > 0)
s.add(y < 5)
s.add(x + y == 7)

print("check:", s.check())
m = s.model()
print(m)
print("x =", m[x])
print("y =", m[y])

check: sat
[y = 0, x = 7]
x = 7
y = 0


### 3.1 一个问题可能有很多解

Z3 默认只返回一个满足条件的解，不保证是最小、最大或唯一。

如果想继续找下一个解，可以加一个“不要等于当前解”的约束。

In [10]:
x = Int("x")
y = Int("y")

s = Solver()
s.add(x >= 0, x <= 5)
s.add(y >= 0, y <= 5)
s.add(x + y == 5)

for i in range(6):
    if s.check() != sat:
        break
    m = s.model()
    xv = m[x].as_long()
    yv = m[y].as_long()
    print(f"solution {i}: x={xv}, y={yv}")

    # 阻止 Z3 下次返回同一个解
    s.add(Or(x != xv, y != yv))

solution 0: x=5, y=0
solution 1: x=4, y=1
solution 2: x=3, y=2
solution 3: x=2, y=3
solution 4: x=1, y=4
solution 5: x=0, y=5


## 4. Bool、And、Or、Not：表达分支条件

程序里的 `if` 条件通常会变成布尔表达式。

```python
if addr == 0x5A and length <= 4:
    ...
```

在 Z3 里类似写成：

```python
And(addr == 0x5A, length <= 4)
```

In [11]:
addr = Int("addr")
length = Int("length")

ok = And(addr == 0x5A, length > 0, length <= 4)

s = Solver()
s.add(ok)

print("check:", s.check())
print(s.model())

check: sat
[length = 1, addr = 90]


## 5. `push()` / `pop()`：临时加约束

调试符号执行时，经常需要：

```text
先保留已有路径约束
临时尝试翻转某个分支
试完再恢复
```

`push()` 会保存当前状态，`pop()` 会回到保存点。

In [12]:
x = Int("x")
s = Solver()
s.add(x >= 0, x <= 10)

print("base:", s.check(), s.model())

s.push()
s.add(x > 100)
print("after temporary impossible constraint:", s.check())
s.pop()

print("after pop:", s.check(), s.model())

base: sat [x = 0]
after temporary impossible constraint: unsat
after pop: sat [x = 0]


## 6. `sexpr()`：看 Z3 内部到底收到了什么

`print(expr)` 适合人看，`sexpr()` 更接近 Z3 内部格式。

CO3 demo 的 log 里会把 `solver.sexpr()` 写出来，方便确认主机真的把约束交给了 Z3。

In [13]:
x = Int("x")
expr = And(x >= 1, x <= 4)

print("python print:", expr)
print("sexpr:")
print(expr.sexpr())

s = Solver()
s.add(expr)
print("solver.sexpr():")
print(s.sexpr())

python print: And(x >= 1, x <= 4)
sexpr:
(and (>= x 1) (<= x 4))
solver.sexpr():
(declare-fun x () Int)
(assert (and (>= x 1) (<= x 4)))



## 7. Int 和 BitVec：CO3 为什么用 BitVec

协议输入通常是字节。一个字节不是无限大的整数，而是固定 8 位：

```text
0x00 ~ 0xFF
```

Z3 里：

- `Int("x")`：数学整数，没有位宽
- `BitVec("b", 8)`：8 位位向量，像 C 里的 `uint8_t`

CO3 demo 用 `BitVec(..., 8)` 来表示协议输入字节。

In [14]:
x = Int("x")
b = BitVec("b", 8)

print("Int x:", x)
print("8-bit BitVec b:", b)

print("255 + 1 as Int:", simplify(IntVal(255) + 1))
print("255 + 1 as 8-bit BitVec:", simplify(BitVecVal(255, 8) + 1))

Int x: x
8-bit BitVec b: b
255 + 1 as Int: 256
255 + 1 as 8-bit BitVec: 0


上面最后一行很关键：8 位字节里 `255 + 1` 会回绕成 `0`。这更接近固件里的字节运算。

## 8. 位向量比较：有符号 vs 无符号

这是新手最容易踩坑的地方。

`BitVec` 只是 8 个 bit，本身不说它是 signed 还是 unsigned。比较时要选对解释方式：

| 写法 | 含义 |
|---|---|
| `x < y` | 有符号比较 |
| `ULT(x, y)` | unsigned less than |
| `ULE(x, y)` | unsigned less or equal |
| `UGT(x, y)` | unsigned greater than |
| `UGE(x, y)` | unsigned greater or equal |

协议字节通常应该按无符号理解，所以 CO3 demo 用 `UGE/ULE`。

In [15]:
b = BitVec("b", 8)

# 0xFF 如果按 unsigned 看是 255；如果按 signed 8-bit 看是 -1。
print("signed  0xFF < 1:", simplify(BitVecVal(0xFF, 8) < BitVecVal(1, 8)))
print("unsigned 0xFF < 1:", simplify(ULT(BitVecVal(0xFF, 8), BitVecVal(1, 8))))

s = Solver()
s.add(UGE(b, 1), ULE(b, 4))
print("range check:", s.check(), s.model())

signed  0xFF < 1: True
unsigned 0xFF < 1: False
range check: sat [b = 4]


## 9. Buffer：一组协议输入字节

在符号执行里，协议输入经常被建模成一个 byte buffer：

```python
input_0, input_1, input_2, ...
```

每个字节都是一个 8-bit `BitVec`。

In [16]:
buf = [BitVec(f"input_{i}", 8) for i in range(4)]

s = Solver()
s.add(buf[0] == 0x5A)
s.add(UGE(buf[1], 1), ULE(buf[1], 4))

print("check:", s.check())
m = s.model()
for i, b in enumerate(buf):
    v = m.eval(b, model_completion=True).as_long()
    print(f"input_{i} = 0x{v:02x}")

check: sat
input_0 = 0x5a
input_1 = 0x04
input_2 = 0x00
input_3 = 0x00


### 9.1 `model_completion=True` 是什么

如果某个变量没有被任何约束限制，Z3 的 model 里可能没有它。

`m.eval(var, model_completion=True)` 的意思是：即使变量没被约束，也给我一个默认值。

CO3 里我们通常只修改被模型明确求出来的字段，其他字段沿用当前输入。

In [17]:
a = BitVec("a", 8)
b = BitVec("b", 8)

s = Solver()
s.add(a == 3)
s.check()
m = s.model()

print("model:", m)
print("m[a]:", m[a])
print("m[b]:", m[b])
print("m.eval(b, model_completion=True):", m.eval(b, model_completion=True))

model: [a = 3]
m[a]: 3
m[b]: None
m.eval(b, model_completion=True): 0


## 10. 路径约束：把一次执行路径翻译成 Z3

假设固件逻辑是：

```python
if addr == 0x5A:
    if 1 <= length <= 4:
        bug()
```

如果当前输入是：

```text
addr = 0x10
length = 0x09
```

设备第一次运行会停在第一个分支：

```text
BRANCH_ADDR = false
```

主机要做的不是修改代码，而是求一个下一轮输入，让这个分支变成 true：

```text
addr == 0x5A
```

In [18]:
input_0 = BitVec("input_0", 8)  # addr
input_1 = BitVec("input_1", 8)  # length

s = Solver()

# 第一次运行：BRANCH_ADDR 是 false。
# 翻转它：下一轮强制 BRANCH_ADDR 为 true。
s.add(input_0 == 0x5A)

print("check:", s.check())
m = s.model()
print("next input_0 =", hex(m[input_0].as_long()))
print("input_1 not in model:", m[input_1])

check: sat
next input_0 = 0x5a
input_1 not in model: None


注意：`input_1` 这时没有出现在 model 里，因为它还没有被任何约束限制。实际 CO3 主机会保留当前的 `length = 0x09`。

## 11. 第二轮：保留已经通过的分支，翻转下一个失败分支

下一轮输入变成：

```text
addr = 0x5A
length = 0x09
```

设备会通过地址检查，然后在长度检查失败：

```text
BRANCH_ADDR = true
BRANCH_LEN  = false
```

主机要做两件事：

```text
保留已经通过的分支：addr == 0x5A
翻转失败的分支：1 <= length <= 4
```

In [19]:
input_0 = BitVec("input_0", 8)
input_1 = BitVec("input_1", 8)

s = Solver()

# keep BRANCH_ADDR=true
s.add(input_0 == 0x5A)

# flip BRANCH_LEN=false -> true
s.add(UGE(input_1, 1), ULE(input_1, 4))

print("solver.sexpr():")
print(s.sexpr())

print("check:", s.check())
m = s.model()
print("next input_0 =", hex(m[input_0].as_long()))
print("next input_1 =", hex(m[input_1].as_long()))

solver.sexpr():
(declare-fun input_0 () (_ BitVec 8))
(declare-fun input_1 () (_ BitVec 8))
(assert (= input_0 #x5a))
(assert (bvuge input_1 #x01))
(assert (bvule input_1 #x04))

check: sat
next input_0 = 0x5a
next input_1 = 0x4


这就是 `host.py` 里的核心思想：

```python
if taken:
    solver.add(make_branch_true)  # 保留已经走过的 true 分支
else:
    solver.add(make_branch_true)  # 把 false 分支翻成 true
    break
```

看起来两边都是 `add(make_branch_true)`，区别在语义：

- `taken == True`：这个分支本来就 true，保留它，保证下一轮还走到这里
- `taken == False`：这个分支本来 false，现在要求它 true，也就是翻转

## 12. `Not(...)` 什么时候用？

如果目标是探索相反方向，通常会用 `Not(branch_condition)`。

例如当前走了：

```text
addr == 0x5A 是 true
```

想探索 else 分支，就加：

```python
Not(addr == 0x5A)
```

但本 CO3 demo 的目标是“深入到 BUG 路径”，所以遇到失败分支时，是把失败的条件强制变成 true，而不是取反成 false。

In [20]:
addr = BitVec("addr", 8)

# 探索 else：要求 addr != 0x5A
s = Solver()
s.add(Not(addr == 0x5A))

print("check:", s.check())
print("example addr =", hex(s.model()[addr].as_long()))

check: sat
example addr = 0xa5


## 13. 从 SVFG 模板到 Z3 表达式

CO3 里主机不是凭空写约束。它依赖编译期提取出的 SVFG 摘要。

本 demo 里用一张表模拟：

```python
EXTRACTED_SVFG = {
    FUNC_CHECK_ADDR:   ("eq_const", "param0", 0x5A),
    FUNC_CHECK_LENGTH: ("range_u8", "param0", 1, 4),
}
```

意思是：

```text
check_addr(param0)   为 true 的条件是 param0 == 0x5A
check_length(param0) 为 true 的条件是 1 <= param0 <= 4
```

运行时 CALL 再告诉主机：

```text
check_addr.param0   <- input_0
check_length.param0 <- input_1
```

合起来才能生成：

```text
input_0 == 0x5A
1 <= input_1 <= 4
```

In [21]:
def build_formula(op_tuple, actual_input):
    op = op_tuple[0]
    if op == "eq_const":
        _, _param, const = op_tuple
        return actual_input == const
    if op == "range_u8":
        _, _param, lo, hi = op_tuple
        return And(UGE(actual_input, lo), ULE(actual_input, hi))
    raise ValueError(op)

input_0 = BitVec("input_0", 8)
input_1 = BitVec("input_1", 8)

svfg_addr = ("eq_const", "param0", 0x5A)
svfg_len = ("range_u8", "param0", 1, 4)

addr_formula = build_formula(svfg_addr, input_0)
len_formula = build_formula(svfg_len, input_1)

print("addr formula:", addr_formula)
print("len formula:", len_formula)

addr formula: input_0 == 90
len formula: And(UGE(input_1, 1), ULE(input_1, 4))


## 14. 完整模拟 CO3 的两次求解

下面这段代码把前面的概念合在一起：

```text
第一轮 trace: BRANCH_ADDR=false
求解: input_0 == 0x5A

第二轮 trace: BRANCH_ADDR=true, BRANCH_LEN=false
求解: input_0 == 0x5A and 1 <= input_1 <= 4
```

In [22]:
def solve_from_branches(branches, current_input):
    input_0 = BitVec("input_0", 8)
    input_1 = BitVec("input_1", 8)

    formulas = {
        "BRANCH_ADDR": input_0 == 0x5A,
        "BRANCH_LEN": And(UGE(input_1, 1), ULE(input_1, 4)),
    }

    s = Solver()
    for branch_name, taken in branches:
        make_true = formulas[branch_name]
        if taken:
            print("keep", branch_name, "true:", make_true)
            s.add(make_true)
        else:
            print("flip", branch_name, "false -> true:", make_true)
            s.add(make_true)
            break

    if s.check() != sat:
        return None

    m = s.model()
    next_input = dict(current_input)
    for idx, var in [(0, input_0), (1, input_1)]:
        value = m[var]
        if value is not None:
            next_input[idx] = value.as_long()
    return next_input

current = {0: 0x10, 1: 0x09}
print("current:", current)

current = solve_from_branches([("BRANCH_ADDR", False)], current)
print("after round 1:", current)

current = solve_from_branches([("BRANCH_ADDR", True), ("BRANCH_LEN", False)], current)
print("after round 2:", current)

current: {0: 16, 1: 9}
flip BRANCH_ADDR false -> true: input_0 == 90
after round 1: {0: 90, 1: 9}
keep BRANCH_ADDR true: input_0 == 90
flip BRANCH_LEN false -> true: And(UGE(input_1, 1), ULE(input_1, 4))
after round 2: {0: 90, 1: 4}


## 15. unsat core：为什么有时没有下一轮输入

如果约束互相冲突，Z3 会返回 `unsat`。调试时可以用 unsat core 看是哪几条约束冲突。

这不是 CO3 最小 demo 必需的，但对理解求解失败很有帮助。

In [23]:
x = Int("x")
s = Solver()

s.assert_and_track(x > 10, "need_large")
s.assert_and_track(x < 5, "need_small")

print("check:", s.check())
print("unsat core:", s.unsat_core())

check: unsat
unsat core: [need_small, need_large]


## 16. 常见坑总结

1. `model` 只是一个可行解，不是唯一答案。
2. `BitVec` 会按位宽回绕，`255 + 1` 在 8 bit 下是 `0`。
3. `BitVec` 的 `<`、`<=` 是有符号比较；协议字节范围通常用 `UGE/ULE/UGT/ULT`。
4. 变量没被约束时，model 里可能没有它。
5. `solver.add(...)` 是不断累加约束，不是覆盖旧约束。
6. CO3 的“翻转分支”不是改设备代码，而是求下一轮输入。
7. SVFG 提供“分支条件模板”，trace 提供“这次运行到了哪里、参数绑定到哪个输入”。

## 17. 练习

可以尝试自己改下面几个条件：

1. 把地址魔数从 `0x5A` 改成 `0xA5`，看 model 怎么变。
2. 把长度范围改成 `2 <= length <= 6`。
3. 加一个第三个分支：`input_2 == input_0 + input_1`。
4. 把 `ULE` 改成普通 `<=`，再试试 `0xFF` 相关条件，观察 signed/unsigned 差异。
5. 在同目录下的 `host.py` 里对照 `flip_first_failed_branch()`，确认 notebook 的第 14 节和真实代码一一对应。